In [26]:
import pandas as pd

df = pd.read_csv('/content/wb_misclassified_detailed.csv')
display(df.head())

,dataset,source_model,method,error_type,score,threshold,misclassified_text,original_text
0,XSUM,perturbation_1_DetectGPT,xsum,FP (Human->AI),1.979690e+15,4.430012e+14,Rangers confirmed that Jardine had lost an 18-...,Rangers confirmed that Jardine had lost an 18-...
1,XSUM,perturbation_1_DetectGPT,xsum,FP (Human->AI),1.419652e+15,4.430012e+14,"The train, which crashed into a pylon as it de...","The train, which crashed into a pylon as it de..."
2,XSUM,perturbation_1_DetectGPT,xsum,FP (Human->AI),6.450729e+14,4.430012e+14,Many growers blame the weak pound which has re...,Many growers blame the weak pound which has re...
3,XSUM,perturbation_1_DetectGPT,xsum,FP (Human->AI),4.972670e+14,4.430012e+14,"The claim: ""Last year alone a city the size of...","The claim: ""Last year alone a city the size of..."
4,XSUM,perturbation_1_DetectGPT,xsum,FP (Human->AI),9.098265e+14,4.430012e+14,"The Scot, 40, has won the ranking tournament t...","The Scot, 40, has won the ranking tournament t..."


In [20]:
import re

def extract_labels_from_error_type(error_type_str):
    # Regex to extract content inside parentheses and split by '->'
    match = re.search(r'\((.*?)\->(.*?)\)', error_type_str)
    if match:
        actual_label = match.group(1)
        predicted_label = match.group(2)
        return actual_label, predicted_label
    return None, None

# Apply the function to create the new columns
df[['Actual_Label', 'Predicted_Label']] = df['error_type'].apply(lambda x: pd.Series(extract_labels_from_error_type(x)))

display(df.head())

,dataset,source_model,method,error_type,score,threshold,misclassified_text,original_text,Actual_Label,Predicted_Label
0,XSUM,perturbation_1_DetectGPT,xsum,FP (Human->AI),1.979690e+15,4.430012e+14,Rangers confirmed that Jardine had lost an 18-...,Rangers confirmed that Jardine had lost an 18-...,Human,AI
1,XSUM,perturbation_1_DetectGPT,xsum,FP (Human->AI),1.419652e+15,4.430012e+14,"The train, which crashed into a pylon as it de...","The train, which crashed into a pylon as it de...",Human,AI
2,XSUM,perturbation_1_DetectGPT,xsum,FP (Human->AI),6.450729e+14,4.430012e+14,Many growers blame the weak pound which has re...,Many growers blame the weak pound which has re...,Human,AI
3,XSUM,perturbation_1_DetectGPT,xsum,FP (Human->AI),4.972670e+14,4.430012e+14,"The claim: ""Last year alone a city the size of...","The claim: ""Last year alone a city the size of...",Human,AI
4,XSUM,perturbation_1_DetectGPT,xsum,FP (Human->AI),9.098265e+14,4.430012e+14,"The Scot, 40, has won the ranking tournament t...","The Scot, 40, has won the ranking tournament t...",Human,AI


In [21]:
# Identify most common misclassification patterns
misclassification_counts = df.groupby(['Actual_Label', 'Predicted_Label']).size().sort_values(ascending=False)
print("\nMost Common Misclassification Patterns:")
display(misclassification_counts.reset_index(name='Count'))

# Calculate total number of misclassified examples
total_misclassified = len(df)

# Calculate percentages for each misclassification pattern
misclassification_percentages = (misclassification_counts / total_misclassified) * 100

print("Percentage of Misclassification Patterns (relative to misclassified samples):")
display(misclassification_percentages.reset_index(name='Percentage'))

print(f"\nTotal misclassified samples: {total_misclassified}")


Most Common Misclassification Patterns:


,Actual_Label,Predicted_Label,Count
0,AI,Human,453
1,Human,AI,422


Percentage of Misclassification Patterns (relative to misclassified samples):


,Actual_Label,Predicted_Label,Percentage
0,AI,Human,51.771429
1,Human,AI,48.228571



Total misclassified samples: 875


In [ ]:
# Install NLTK for text analysis
!pip install nltk
import nltk
nltk.download('punkt')
nltk.download('punkt_tab') # Corrected: Added download for 'punkt_tab'

Next, let's explore the linguistic patterns. I'll add cells to set up the necessary libraries and then extract features like the number of sentences and words from the `misclassified_text` and `original_text`.

In [24]:
from nltk.tokenize import sent_tokenize, word_tokenize

def count_sentences(text):
    if pd.isna(text):
        return 0
    return len(sent_tokenize(str(text)))

def count_words(text):
    if pd.isna(text):
        return 0
    return len(word_tokenize(str(text)))

# Apply linguistic feature extraction
df['original_sentence_count'] = df['original_text'].apply(count_sentences)
df['original_word_count'] = df['original_text'].apply(count_words)
df['misclassified_sentence_count'] = df['misclassified_text'].apply(count_sentences)
df['misclassified_word_count'] = df['misclassified_text'].apply(count_words)

# Display the DataFrame with new linguistic features for initial inspection
print("DataFrame with new linguistic features:")
display(df[['error_type', 'original_text', 'original_sentence_count', 'original_word_count', 'misclassified_text', 'misclassified_sentence_count', 'misclassified_word_count']].head())

DataFrame with new linguistic features:


,error_type,original_text,original_sentence_count,original_word_count,misclassified_text,misclassified_sentence_count,misclassified_word_count
0,FP (Human->AI),Rangers confirmed that Jardine had lost an 18-...,7,184,Rangers confirmed that Jardine had lost an 18-...,7,184
1,FP (Human->AI),"The train, which crashed into a pylon as it de...",7,178,"The train, which crashed into a pylon as it de...",7,178
2,FP (Human->AI),Many growers blame the weak pound which has re...,11,194,Many growers blame the weak pound which has re...,11,194
3,FP (Human->AI),"The claim: ""Last year alone a city the size of...",4,132,"The claim: ""Last year alone a city the size of...",4,132
4,FP (Human->AI),"The Scot, 40, has won the ranking tournament t...",9,194,"The Scot, 40, has won the ranking tournament t...",9,194


In [25]:
# Group by error_type and calculate the mean of linguistic features
linguistic_patterns = df.groupby('error_type')[['original_sentence_count', 'original_word_count', 'misclassified_sentence_count', 'misclassified_word_count']].mean().round(2)

print("Average Linguistic Features per Misclassification Type (based on error_type):")
display(linguistic_patterns)

# Also consider the patterns by Actual and Predicted Labels for clearer interpretation
linguistic_patterns_by_labels = df.groupby(['Actual_Label', 'Predicted_Label'])[['original_sentence_count', 'original_word_count', 'misclassified_sentence_count', 'misclassified_word_count']].mean().round(2)

print("\nAverage Linguistic Features for (Actual -> Predicted) Misclassification:")
display(linguistic_patterns_by_labels)

Average Linguistic Features per Misclassification Type (based on error_type):


,original_sentence_count,original_word_count,misclassified_sentence_count,misclassified_word_count
error_type,,,,
FN (AI->Human),9.92,183.83,9.14,182.83
FP (Human->AI),8.88,184.12,8.88,184.12



Average Linguistic Features for (Actual -> Predicted) Misclassification:


,,original_sentence_count,original_word_count,misclassified_sentence_count,misclassified_word_count
Actual_Label,Predicted_Label,,,,
AI,Human,9.92,183.83,9.14,182.83
Human,AI,8.88,184.12,8.88,184.12


In [27]:
import pandas as pd

df = pd.read_csv('/content/updated_misclassified_data.csv')
display(df.head())

,dataset,source_model,method,error_type,score,threshold,misclassified_text,original_text,Actual_Label,Predicted_Label,Linguistic_Pattern_Category,Percentage_of_Misclassification_Type
0,XSUM,perturbation_1_DetectGPT,xsum,FP (Human->AI),1.979690e+15,4.430012e+14,Rangers confirmed that Jardine had lost an 18-...,Rangers confirmed that Jardine had lost an 18-...,Human,AI,Human -> AI (average sentence length),33.371429
1,XSUM,perturbation_1_DetectGPT,xsum,FP (Human->AI),1.419652e+15,4.430012e+14,"The train, which crashed into a pylon as it de...","The train, which crashed into a pylon as it de...",Human,AI,Human -> AI (average sentence length),33.371429
2,XSUM,perturbation_1_DetectGPT,xsum,FP (Human->AI),6.450729e+14,4.430012e+14,Many growers blame the weak pound which has re...,Many growers blame the weak pound which has re...,Human,AI,Human -> AI (average sentence length),33.371429
3,XSUM,perturbation_1_DetectGPT,xsum,FP (Human->AI),4.972670e+14,4.430012e+14,"The claim: ""Last year alone a city the size of...","The claim: ""Last year alone a city the size of...",Human,AI,Human -> AI (simple sentence),9.828571
4,XSUM,perturbation_1_DetectGPT,xsum,FP (Human->AI),9.098265e+14,4.430012e+14,"The Scot, 40, has won the ranking tournament t...","The Scot, 40, has won the ranking tournament t...",Human,AI,Human -> AI (average sentence length),33.371429


In [11]:
print("Individual Misclassification and Linguistic Patterns per Row (first 5 examples):")
display(df[['original_text', 'misclassified_text', 'Actual_Label', 'Predicted_Label',
            'original_sentence_count', 'original_word_count',
            'misclassified_sentence_count', 'misclassified_word_count']].head())

Individual Misclassification and Linguistic Patterns per Row (first 5 examples):


,original_text,misclassified_text,Actual_Label,Predicted_Label,original_sentence_count,original_word_count,misclassified_sentence_count,misclassified_word_count
0,Rangers confirmed that Jardine had lost an 18-...,Rangers confirmed that Jardine had lost an 18-...,Human,AI,7,184,7,184
1,"The train, which crashed into a pylon as it de...","The train, which crashed into a pylon as it de...",Human,AI,7,178,7,178
2,Many growers blame the weak pound which has re...,Many growers blame the weak pound which has re...,Human,AI,11,194,11,194
3,"The claim: ""Last year alone a city the size of...","The claim: ""Last year alone a city the size of...",Human,AI,4,132,4,132
4,"The Scot, 40, has won the ranking tournament t...","The Scot, 40, has won the ranking tournament t...",Human,AI,9,194,9,194


### Categorizing Linguistic Patterns and Calculating Their Percentages

To fulfill your request, I will categorize each misclassified entry into specific linguistic patterns based on the `original_sentence_count`. For this, I'll use the following thresholds:

*   **Simple Sentence:** `original_sentence_count <= 6`
*   **Longer Sentence:** `original_sentence_count >= 13`
*   **Average Sentence Length:** Anything in between

These categories will be combined with the `Actual_Label` and `Predicted_Label` to form a `Linguistic_Pattern_Category` (e.g., 'Human -> AI (simple sentence)'). I will then calculate the percentage of each of these combined patterns relative to the total number of misclassified samples and add this as a new column, `Percentage_of_Misclassification_Type`, to the DataFrame.


In [12]:
def create_detailed_linguistic_pattern(row):
    actual = row['Actual_Label']
    predicted = row['Predicted_Label']
    sentence_count = row['original_sentence_count']

    # Define thresholds for sentence complexity
    simple_sentence_threshold = 6
    longer_sentence_threshold = 13

    sentence_description = ""
    if sentence_count <= simple_sentence_threshold:
        sentence_description = "(simple sentence)"
    elif sentence_count >= longer_sentence_threshold:
        sentence_description = "(longer sentence)"
    else:
        sentence_description = "(average sentence length)"

    return f"{actual} -> {predicted} {sentence_description}"

# Apply the function to create the new 'Linguistic_Pattern_Category' column
df['Linguistic_Pattern_Category'] = df.apply(create_detailed_linguistic_pattern, axis=1)

print("DataFrame with new 'Linguistic_Pattern_Category' column:")
display(df[['original_text', 'Actual_Label', 'Predicted_Label', 'original_sentence_count', 'Linguistic_Pattern_Category']].head())


DataFrame with new 'Linguistic_Pattern_Category' column:


,original_text,Actual_Label,Predicted_Label,original_sentence_count,Linguistic_Pattern_Category
0,Rangers confirmed that Jardine had lost an 18-...,Human,AI,7,Human -> AI (average sentence length)
1,"The train, which crashed into a pylon as it de...",Human,AI,7,Human -> AI (average sentence length)
2,Many growers blame the weak pound which has re...,Human,AI,11,Human -> AI (average sentence length)
3,"The claim: ""Last year alone a city the size of...",Human,AI,4,Human -> AI (simple sentence)
4,"The Scot, 40, has won the ranking tournament t...",Human,AI,9,Human -> AI (average sentence length)


In [15]:
df = df.drop(columns=['original_word_count', 'misclassified_sentence_count', 'misclassified_word_count'])

print("DataFrame after removing specified linguistic feature columns (first 5 examples):")
display(df.head())

DataFrame after removing specified linguistic feature columns (first 5 examples):


,dataset,source_model,method,error_type,score,threshold,misclassified_text,original_text,Actual_Label,Predicted_Label,Linguistic_Pattern_Category,Percentage_of_Misclassification_Type
0,XSUM,perturbation_1_DetectGPT,xsum,FP (Human->AI),1.979690e+15,4.430012e+14,Rangers confirmed that Jardine had lost an 18-...,Rangers confirmed that Jardine had lost an 18-...,Human,AI,Human -> AI (average sentence length),33.371429
1,XSUM,perturbation_1_DetectGPT,xsum,FP (Human->AI),1.419652e+15,4.430012e+14,"The train, which crashed into a pylon as it de...","The train, which crashed into a pylon as it de...",Human,AI,Human -> AI (average sentence length),33.371429
2,XSUM,perturbation_1_DetectGPT,xsum,FP (Human->AI),6.450729e+14,4.430012e+14,Many growers blame the weak pound which has re...,Many growers blame the weak pound which has re...,Human,AI,Human -> AI (average sentence length),33.371429
3,XSUM,perturbation_1_DetectGPT,xsum,FP (Human->AI),4.972670e+14,4.430012e+14,"The claim: ""Last year alone a city the size of...","The claim: ""Last year alone a city the size of...",Human,AI,Human -> AI (simple sentence),9.828571
4,XSUM,perturbation_1_DetectGPT,xsum,FP (Human->AI),9.098265e+14,4.430012e+14,"The Scot, 40, has won the ranking tournament t...","The Scot, 40, has won the ranking tournament t...",Human,AI,Human -> AI (average sentence length),33.371429


In [16]:
from google.colab import files

# Save the DataFrame to a CSV file
df.to_csv('updated_misclassified_data.csv', index=False)

# Download the file to your local machine
files.download('updated_misclassified_data.csv')

print("Your updated CSV file 'updated_misclassified_data.csv' should now be downloaded to your desktop.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Your updated CSV file 'updated_misclassified_data.csv' should now be downloaded to your desktop.


In [13]:
# Calculate the count of each unique linguistic pattern category
pattern_counts = df['Linguistic_Pattern_Category'].value_counts()

# Calculate the total number of misclassified samples (already known as `total_misclassified` from previous steps)
total_misclassified = len(df)

# Calculate the percentage of each pattern within the total misclassified samples
pattern_percentages = (pattern_counts / total_misclassified) * 100

# Convert to a DataFrame for easier mapping
percentage_df = pattern_percentages.rename('Percentage_of_Misclassification_Type').reset_index()
percentage_df.rename(columns={'index': 'Linguistic_Pattern_Category'}, inplace=True)

# Map the percentages back to the original DataFrame
df = df.merge(percentage_df, on='Linguistic_Pattern_Category', how='left')

print("DataFrame with 'Percentage_of_Misclassification_Type' column (first 5 examples):")
display(df[['original_text', 'Actual_Label', 'Predicted_Label',
            'original_sentence_count', 'original_word_count',
            'Linguistic_Pattern_Category', 'Percentage_of_Misclassification_Type']].head())

print("\nSummary of Linguistic Pattern Categories and their Percentages:")
display(percentage_df)


DataFrame with 'Percentage_of_Misclassification_Type' column (first 5 examples):


,original_text,Actual_Label,Predicted_Label,original_sentence_count,original_word_count,Linguistic_Pattern_Category,Percentage_of_Misclassification_Type
0,Rangers confirmed that Jardine had lost an 18-...,Human,AI,7,184,Human -> AI (average sentence length),33.371429
1,"The train, which crashed into a pylon as it de...",Human,AI,7,178,Human -> AI (average sentence length),33.371429
2,Many growers blame the weak pound which has re...,Human,AI,11,194,Human -> AI (average sentence length),33.371429
3,"The claim: ""Last year alone a city the size of...",Human,AI,4,132,Human -> AI (simple sentence),9.828571
4,"The Scot, 40, has won the ranking tournament t...",Human,AI,9,194,Human -> AI (average sentence length),33.371429



Summary of Linguistic Pattern Categories and their Percentages:


,Linguistic_Pattern_Category,Percentage_of_Misclassification_Type
0,AI -> Human (average sentence length),33.714286
1,Human -> AI (average sentence length),33.371429
2,Human -> AI (simple sentence),9.828571
3,AI -> Human (longer sentence),9.600000
4,AI -> Human (simple sentence),8.457143
5,Human -> AI (longer sentence),5.028571


This table shows the individual breakdown for each misclassified entry. If you need to explore specific ranges or filter for certain patterns, you can use standard pandas DataFrame operations.

As a reminder, to get the **overall misclassification percentage** for the entire dataset, we still need the **total number of samples (N_total)** from your complete original dataset.

In [9]:
# Group by error_type and calculate the mean of linguistic features
linguistic_patterns = df.groupby('error_type')[['original_sentence_count', 'original_word_count', 'misclassified_sentence_count', 'misclassified_word_count']].mean().round(2)

print("Average Linguistic Features per Misclassification Type:")
display(linguistic_patterns)

# Also consider the patterns by Actual and Predicted Labels for clearer interpretation
linguistic_patterns_by_labels = df.groupby(['Actual_Label', 'Predicted_Label'])[['original_sentence_count', 'original_word_count', 'misclassified_sentence_count', 'misclassified_word_count']].mean().round(2)

print("\nAverage Linguistic Features for (Actual -> Predicted) Misclassification:")
display(linguistic_patterns_by_labels)


Average Linguistic Features per Misclassification Type:


,original_sentence_count,original_word_count,misclassified_sentence_count,misclassified_word_count
error_type,,,,
FN (AI->Human),9.92,183.83,9.14,182.83
FP (Human->AI),8.88,184.12,8.88,184.12



Average Linguistic Features for (Actual -> Predicted) Misclassification:


,,original_sentence_count,original_word_count,misclassified_sentence_count,misclassified_word_count
Actual_Label,Predicted_Label,,,,
AI,Human,9.92,183.83,9.14,182.83
Human,AI,8.88,184.12,8.88,184.12
